In [1]:
import os
import cv2
import time
import timm
import torch
import random
import numpy as np
import pandas as pd

from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

import albumentations as A
from albumentations.pytorch import ToTensorV2

from tqdm.auto import tqdm


d:\BDC_2026_PP\P_PP\poetri\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
CFG = {

    "model_name":
        "vit_base_patch16_siglip_384",

    "img_size":384,

    "batch_size":16,

    "epochs":25,

    "patience":4,

    "lr":5e-5,

    "weight_decay":1e-4,

    "num_classes":3,

    "num_workers":0,

    "device":"cuda",

    "folds":5
}



In [4]:
train_transform = A.Compose([

    A.LongestMaxSize(
        max_size=CFG["img_size"]
    ),

    A.PadIfNeeded(
        min_height=CFG["img_size"],
        min_width=CFG["img_size"],
        border_mode=cv2.BORDER_CONSTANT
    ),

    A.HorizontalFlip(p=0.5),

    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.10,
        rotate_limit=15,
        p=0.5
    ),

    A.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1,
        p=0.5
    ),

    A.HueSaturationValue(
        hue_shift_limit=10,
        sat_shift_limit=15,
        val_shift_limit=10,
        p=0.5
    ),

    A.CoarseDropout(
        num_holes_range=(1,3),
        hole_height_range=(0.05,0.15),
        hole_width_range=(0.05,0.15),
        p=0.5
    ),

    A.Normalize(),

    ToTensorV2()

])


d:\BDC_2026_PP\P_PP\poetri\.venv\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [5]:
valid_transform = A.Compose([

    A.LongestMaxSize(
        max_size=CFG["img_size"]
    ),

    A.PadIfNeeded(
        min_height=CFG["img_size"],
        min_width=CFG["img_size"],
        border_mode=cv2.BORDER_CONSTANT
    ),

    A.Normalize(),

    ToTensorV2()

])


In [6]:
df = pd.read_csv("train_folds.csv")

print(df.shape)
display(df.head())


(24853, 4)


,filepath,label,class_name,fold
0,..\data\train\0_Recyclable\R_1.jpg,0,0_Recyclable,2
1,..\data\train\0_Recyclable\R_10.jpg,0,0_Recyclable,2
2,..\data\train\0_Recyclable\R_100.jpg,0,0_Recyclable,2
3,..\data\train\0_Recyclable\R_1000.jpg,0,0_Recyclable,3
4,..\data\train\0_Recyclable\R_1001.jpg,0,0_Recyclable,3


In [7]:
class WasteDataset(Dataset):

    def __init__(self, dataframe, transforms=None):

        self.df = dataframe.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = cv2.imread(row.filepath)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms is not None:
            image = self.transforms(image=image)["image"]

        label = int(row.label)

        return image, label


In [8]:
def build_model():

    model = timm.create_model(

        CFG["model_name"],

        pretrained=True,

        num_classes=CFG["num_classes"]

    )

    return model.to(CFG["device"])


In [9]:
def build_dataloader(fold):

    train_df = df[df.fold != fold].reset_index(drop=True)

    valid_df = df[df.fold == fold].reset_index(drop=True)

    train_dataset = WasteDataset(

        train_df,

        train_transform

    )

    valid_dataset = WasteDataset(

        valid_df,

        valid_transform

    )

    train_loader = DataLoader(

        train_dataset,

        batch_size=CFG["batch_size"],

        shuffle=True,

        num_workers=CFG["num_workers"],

        pin_memory=True,

        persistent_workers=CFG["num_workers"] > 0

    )

    valid_loader = DataLoader(

        valid_dataset,

        batch_size=CFG["batch_size"],

        shuffle=False,

        num_workers=CFG["num_workers"],

        pin_memory=True,

        persistent_workers=CFG["num_workers"] > 0

    )

    return (

        train_df,

        valid_df,

        train_loader,

        valid_loader

    )


In [10]:
def build_class_weight(train_df):

    weights = compute_class_weight(

        class_weight="balanced",

        classes=np.array([0,1,2]),

        y=train_df.label

    )

    weights = torch.tensor(

        weights,

        dtype=torch.float32,

        device=CFG["device"]

    )

    return weights


In [11]:
def build_training_objects(model, class_weights):

    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=0.05
    )

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=CFG["lr"],

        weight_decay=CFG["weight_decay"]

    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(

        optimizer,

        T_max=CFG["epochs"]

    )

    scaler = torch.amp.GradScaler("cuda")

    return (

        criterion,

        optimizer,

        scheduler,

        scaler

    )


In [12]:
def train_one_epoch(

    model,
    loader,
    criterion,
    optimizer,
    scaler

):

    model.train()

    running_loss = 0

    pbar = tqdm(
        loader,
        leave=False,
        desc="Train"
    )

    for images, labels in pbar:

        images = images.to(
            CFG["device"],
            non_blocking=True
        )

        labels = labels.to(
            CFG["device"],
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda"
        ):

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item()

        pbar.set_postfix(

            loss=f"{loss.item():.4f}",

            lr=f"{optimizer.param_groups[0]['lr']:.2e}"

        )

    return running_loss / len(loader)


In [13]:
def valid_one_epoch(

    model,
    loader,
    criterion

):

    model.eval()

    running_loss = 0

    preds = []

    labels_all = []

    pbar = tqdm(
        loader,
        leave=False,
        desc="Valid"
    )

    with torch.no_grad():

        for images, labels in pbar:

            images = images.to(
                CFG["device"],
                non_blocking=True
            )

            labels = labels.to(
                CFG["device"],
                non_blocking=True
            )

            with torch.amp.autocast(
                device_type="cuda"
            ):

                outputs = model(images)

                loss = criterion(
                    outputs,
                    labels
                )

            running_loss += loss.item()

            pred = outputs.argmax(dim=1)

            preds.extend(
                pred.cpu().numpy()
            )

            labels_all.extend(
                labels.cpu().numpy()
            )

            pbar.set_postfix(

                loss=f"{loss.item():.4f}"

            )

    macro_f1 = f1_score(

        labels_all,

        preds,

        average="macro"

    )

    accuracy = (

        np.array(preds) ==
        np.array(labels_all)

    ).mean()

    return (

        running_loss / len(loader),

        accuracy,

        macro_f1

    )


In [14]:
def train_fold(fold):

    print("="*60)
    print(f"Fold {fold}")
    print("="*60)

    train_df, valid_df, train_loader, valid_loader = build_dataloader(fold)

    class_weights = build_class_weight(train_df)

    model = build_model()

    criterion, optimizer, scheduler, scaler = build_training_objects(
        model,
        class_weights
    )

    best_f1 = 0.0
    best_acc = 0.0
    patience_counter = 0

    history = []

    for epoch in range(CFG["epochs"]):

        start = time.time()

        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler
        )

        valid_loss, valid_acc, valid_f1 = valid_one_epoch(
            model=model,
            loader=valid_loader,
            criterion=criterion
        )

        scheduler.step()

        elapsed = time.time() - start

        history.append({

            "epoch": epoch + 1,

            "train_loss": train_loss,

            "valid_loss": valid_loss,

            "valid_acc": valid_acc,

            "valid_f1": valid_f1

        })

        print(
            f"Epoch {epoch+1:02d}/{CFG['epochs']} | "
            f"Train {train_loss:.4f} | "
            f"Valid {valid_loss:.4f} | "
            f"Acc {valid_acc:.4f} | "
            f"F1 {valid_f1:.5f} | "
            f"{elapsed:.1f}s"
        )

        if valid_f1 > best_f1:

            best_f1 = valid_f1
            best_acc = valid_acc

            patience_counter = 0

            torch.save(

                model.state_dict(),

                f"siglip_best_fold{fold}.pth"

            )

            print(
                f"Best Model Saved "
                f"(F1={best_f1:.5f})"
            )

        else:

            patience_counter += 1

            print(
                f"No Improvement "
                f"({patience_counter}/{CFG['patience']})"
            )

        if patience_counter >= CFG["patience"]:

            print("Early Stopping")

            break

    torch.save(

        model.state_dict(),

        f"siglip_last_fold{fold}.pth"

    )

    print()

    print("="*60)

    print(f"Fold {fold} Finished")

    print(f"Best F1  : {best_f1:.5f}")

    print(f"Best Acc : {best_acc:.5f}")

    print("="*60)

    return history


In [15]:
all_history = []

best_scores = []

start_training = time.time()

for fold in range(CFG["folds"]):

    fold_history = train_fold(fold)

    all_history.append(fold_history)

    fold_best_f1 = max(
        [x["valid_f1"] for x in fold_history]
    )

    best_scores.append(fold_best_f1)

total_time = time.time() - start_training


Fold 0


Epoch 01/25 | Train 0.4593 | Valid 0.3665 | Acc 0.9407 | F1 0.93843 | 2127.2s
Best Model Saved (F1=0.93843)


Epoch 02/25 | Train 0.3598 | Valid 0.3797 | Acc 0.9336 | F1 0.93085 | 2121.6s
No Improvement (1/4)


Epoch 03/25 | Train 0.3370 | Valid 0.4078 | Acc 0.9205 | F1 0.91406 | 2147.7s
No Improvement (2/4)


Epoch 04/25 | Train 0.3233 | Valid 0.3669 | Acc 0.9392 | F1 0.93436 | 2275.7s
No Improvement (3/4)


Epoch 05/25 | Train 0.3133 | Valid 0.3594 | Acc 0.9431 | F1 0.94656 | 2121.9s
Best Model Saved (F1=0.94656)


... (91 outputs left out)

Epoch 22/25 | Train 0.2237 | Valid 0.3646 | Acc 0.9515 | F1 0.95511 | 586.2s
No Improvement (1/4)


Epoch 23/25 | Train 0.2234 | Valid 0.3625 | Acc 0.9519 | F1 0.95615 | 586.1s
No Improvement (2/4)


Epoch 24/25 | Train 0.2218 | Valid 0.3631 | Acc 0.9529 | F1 0.95709 | 583.0s
Best Model Saved (F1=0.95709)


Epoch 25/25 | Train 0.2202 | Valid 0.3650 | Acc 0.9513 | F1 0.95518 | 582.9s
No Improvement (1/4)

Fold 4 Finished
Best F1  : 0.95709
Best Acc : 0.95292


In [16]:
print()

print("="*70)
print("TRAINING FINISHED")
print("="*70)

for i, score in enumerate(best_scores):

    print(
        f"Fold {i} : {score:.5f}"
    )

print("-"*70)

print(
    f"Average Macro F1 : "
    f"{np.mean(best_scores):.5f}"
)

print(
    f"Std               : "
    f"{np.std(best_scores):.5f}"
)

print("-"*70)

hours = total_time // 3600
minutes = (total_time % 3600) // 60
seconds = total_time % 60

print(
    f"Training Time : "
    f"{int(hours)}h "
    f"{int(minutes)}m "
    f"{seconds:.0f}s"
)

print("="*70)



TRAINING FINISHED
Fold 0 : 0.96822
Fold 1 : 0.97333
Fold 2 : 0.97974
Fold 3 : 0.95614
Fold 4 : 0.95709
----------------------------------------------------------------------
Average Macro F1 : 0.96690
Std               : 0.00917
----------------------------------------------------------------------
Training Time : 36h 19m 48s
